# L03: Exploratory statistical analysis

**Day 1.** Distributions in operational context, hypothesis tests, ANOVA with post-hoc comparisons, the bootstrap, and the difference between statistical and operational significance. SciPy and statsmodels.

- Reads: checkpoints/L02_clean.csv
- Writes: work/L03_findings.json

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["checkpoints/L02_clean.csv"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats


def skew_summary(x):
    """Mean, median, skewness and mean-to-median ratio of a numeric series.

    Lab L03.
    """
    x = pd.Series(x).dropna()
    return {"mean": float(x.mean()), "median": float(x.median()), "skew": float(stats.skew(x)),
            "mean_over_median": float(x.mean() / x.median())}


def fit_mtbf(gaps_days):
    """Fit an exponential to times between failures; return MTBF in days and the daily rate.

    The maximum-likelihood exponential rate is 1 / (mean gap).

    Lab L03.
    """
    gaps = np.asarray(gaps_days, dtype=float)
    mtbf = float(gaps.mean())
    return {"mtbf_days": mtbf, "rate_per_day": 1 / mtbf, "n_gaps": int(gaps.size)}


def group_compare(df, value, group):
    """One-way ANOVA of `value` across `group`, with Tukey HSD post-hoc comparisons.

    Returns (anova_table, tukey_table) as DataFrames.

    Lab L03.
    """
    from statsmodels.formula.api import ols
    from statsmodels.stats.anova import anova_lm
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    data = df[[value, group]].dropna()
    anova = anova_lm(ols(f"{value} ~ C({group})", data=data).fit())
    tukey = pairwise_tukeyhsd(data[value], data[group])
    rows = tukey.summary().data
    return anova, pd.DataFrame(rows[1:], columns=rows[0])


def bootstrap_ci(a, b, stat=None, n_resamples=2000, level=0.95, seed=2026):
    """Bootstrap confidence interval for stat(a) - stat(b), resampling each group with replacement.

    `stat` is any function of one array; the default, None, means the median.

    Bounded (n_resamples fixed) and seeded, so it returns the same interval every run.

    Lab L03.
    """
    stat = np.median if stat is None else stat
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    diffs = np.array([stat(rng.choice(a, a.size)) - stat(rng.choice(b, b.size)) for _ in range(n_resamples)])
    tail = (1 - level) / 2
    return {"estimate": float(stat(a) - stat(b)),
            "low": float(np.quantile(diffs, tail)), "high": float(np.quantile(diffs, 1 - tail))}

print("Helper functions ready")

## Part 1: Distribution choice

**Specification.** Describe ground time before departure, fit three candidate distributions, and choose the one the
data supports. Then model the time between maintenance delays on the same aircraft.

### Step 3 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import json

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

df = pd.read_csv(input_path("L02_clean.csv"), encoding="utf-8", parse_dates=["departure_date"])
ground = df["ground_time_hours"]
summary = skew_summary(ground)
print(f"ground time: mean {summary['mean']:.1f} h, median {summary['median']:.1f} h, skew {summary['skew']:.2f}")

# Fit each candidate by maximum likelihood and compare: AIC (lower is better) and the
# Kolmogorov-Smirnov distance between the fitted and observed distributions (lower is better).
candidates = {"normal": stats.norm, "exponential": stats.expon, "lognormal": stats.lognorm}
rows = []
for name, dist in candidates.items():
    params = dist.fit(ground) if name == "normal" else dist.fit(ground, floc=0)   # durations start at 0
    n_free = len(params) - (0 if name == "normal" else 1)                          # loc fixed for two of them
    log_lik = dist.logpdf(ground, *params).sum()
    rows.append({"distribution": name, "AIC": round(2 * n_free - 2 * log_lik, 1),
                 "KS distance": round(stats.kstest(ground, dist.cdf, args=params).statistic, 3)})
fits = pd.DataFrame(rows).set_index("distribution")
fits

### Step 4 · DECIDE

In [ ]:
# DECIDE · DP-L03-1 · which distribution the data supports.
# Replace ____ with your choice. The choices are taught on the slide "Choosing a distribution, in practice".
# Compare the three fitted curves and their scores above, then choose.
CHOSEN = "____"

params = candidates[CHOSEN].fit(ground, floc=0) if CHOSEN != "normal" else candidates[CHOSEN].fit(ground)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(ground, bins=60, density=True, alpha=0.5, label="observed")
grid = pd.Series(range(1, 151))
ax.plot(grid, candidates[CHOSEN].pdf(grid, *params), label=f"fitted {CHOSEN}")
ax.axvline(summary["median"], linestyle="--", label="median")
ax.axvline(summary["mean"], linestyle=":", label="mean")
ax.set_xlabel("ground time before departure (hours)")
ax.set_xlim(0, 100)
ax.legend()
plt.show()

### Step 5 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
# Time between failures: for each aircraft, the days between one maintenance delay and the next.
# If delays arrive at a steady rate independent of the past, those gaps are exponential, and the
# mean gap is the mean time between failures (MTBF).
delays = df[df["maint_delay"] == 1].sort_values("departure_date")
gaps = delays.groupby("tail_id")["departure_date"].diff().dt.days.dropna()
gaps = gaps[gaps > 0]
mtbf = fit_mtbf(gaps)
print(f"{mtbf['n_gaps']} gaps; MTBF {mtbf['mtbf_days']:.1f} days; "
      f"coefficient of variation {gaps.std() / gaps.mean():.2f} (an exponential has 1)")
p_week = 1 - stats.expon.cdf(7, scale=mtbf["mtbf_days"])
print(f"chance an aircraft goes a week without a maintenance delay: {p_week:.2f}")

### Step 6 · WRITE

**WRITE · your interpretation**. Why is your choice the right distribution for ground time, and which single number would you
report as a typical ground time? What does the exponential fit to delay gaps let a planner say?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Hypothesis test

**Specification.** C-17A crews at Carver Field report longer ground times than at Halstead AFB. State the hypothesis
before looking at a p-value, then test it. Also test whether delay rates differ across aircraft types.

### Step 7 · WRITE

**WRITE · Your hypotheses**. Before any test runs, write the null hypothesis and the alternative for the C-17A ground-time claim,
in words: what you are testing, on which scale, and between which two bases.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 8 · DECIDE

In [ ]:
# DECIDE · DP-L03-2 · the significance level.
# Replace ____ with your choice. The choices are taught on the slide "Choosing the significance level".
# The hypotheses you wrote above, as the code will print them. The significance level is yours to set.
H0 = "C-17A ground time has the same mean (on the log scale) at Carver Field and Halstead AFB"
H1 = "C-17A ground time has a different mean (on the log scale) at Carver Field and Halstead AFB"
ALPHA = ____

print("H0:", H0)
print("H1:", H1)

### Step 9 · PREDICT

**PREDICT · before you run the next cell**. Before running: with about 500 C-17A departures at each base, do you expect a significant result
even if the true difference is small? Why?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 10 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import numpy as np

c17 = df[df["aircraft_type"] == "C-17A"]
carver = c17.loc[c17["home_base"] == "Carver Field", "ground_time_hours"]
halstead = c17.loc[c17["home_base"] == "Halstead AFB", "ground_time_hours"]
# Welch's t-test on log ground time: the log makes the skewed data close to normal,
# and Welch's version does not assume equal variances.
t = stats.ttest_ind(np.log(carver), np.log(halstead), equal_var=False)
print(f"n = {len(carver)} and {len(halstead)}; t = {t.statistic:.2f}, p = {t.pvalue:.4f}; "
      f"{'reject' if t.pvalue < ALPHA else 'do not reject'} H0 at alpha {ALPHA}")

# Chi-square test of independence: is the delay rate the same for every aircraft type?
table = pd.crosstab(df["aircraft_type"], df["maint_delay"])
chi2, p_chi, dof, _ = stats.chi2_contingency(table)
print(f"delay rate by type: {df.groupby('aircraft_type')['maint_delay'].mean().round(3).to_dict()}")
print(f"chi-square = {chi2:.1f}, dof = {dof}, p = {p_chi:.1e}")

### Step 11 · WRITE

**WRITE · your interpretation**. In this test, what would a Type I error be, and what would a Type II error be?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 3: ANOVA and post-hoc comparisons

**Specification.** Compare ground time across all four aircraft types at once, find which pairs differ, and put a
bootstrap confidence interval on the C-17A base difference.

### Step 12 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
df["log_ground"] = np.log(df["ground_time_hours"])
anova, tukey = group_compare(df, "log_ground", "aircraft_type")
print(anova.to_string(float_format=lambda v: f"{v:.4g}"))
tukey

### Step 13 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
# Bootstrap: resample each base's departures with replacement 2,000 times and look at the spread of
# the median difference. No distributional assumption is needed.
ci = bootstrap_ci(carver, halstead, stat=np.median, n_resamples=2000, seed=SEED)
print(f"median difference, Carver minus Halstead: {ci['estimate']:.2f} h, "
      f"95% interval {ci['low']:.2f} to {ci['high']:.2f} h")

### Step 14 · WRITE

**WRITE · your interpretation**. Why run one ANOVA with post-hoc comparisons instead of six separate t-tests? What does the
bootstrap interval add to the t-test in Part 2?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 4: Operational judgment

**Specification.** Decide whether the C-17A base difference matters to a planner.

### Step 15 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
typical = c17["ground_time_hours"].median()
print(f"typical C-17A ground time: {typical:.1f} h")
print(f"base difference: {ci['estimate']:.2f} h ({ci['estimate'] / typical:.0%} of a typical ground time), "
      f"p = {t.pvalue:.4f}")

### Step 16 · WRITE

**WRITE · Closing operational judgment**. Is the base difference statistically significant, and is it operationally significant? What, if
anything, should the group do about it?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 17 · RUN

In [ ]:
# RUN · saves this part's output to work/.
findings = {
    "ground_time": {"distribution": CHOSEN, "median_hours": round(summary["median"], 2),
                    "mean_hours": round(summary["mean"], 2)},
    "mtbf_days": round(mtbf["mtbf_days"], 2),
    "c17_base_test": {"p_value": round(float(t.pvalue), 5), "median_difference_hours": round(ci["estimate"], 2),
                      "ci_low": round(ci["low"], 2), "ci_high": round(ci["high"], 2)},
    "delay_rate_by_type": df.groupby("aircraft_type")["maint_delay"].mean().round(4).to_dict(),
    "departures_by_type": df["aircraft_type"].value_counts().sort_index().to_dict(),
}
(WORK / "L03_findings.json").write_text(json.dumps(findings, indent=2), encoding="utf-8")
print("saved work/L03_findings.json")